<a href="https://colab.research.google.com/github/mke27/ECON3916-Statistical-Machine-Learning/blob/main/Lab%2012/Lab12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/mke27/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_ZHVI_2026_Micro%20(1).csv'
df = pd.read_csv(url)

In [ ]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [ ]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula,data=df)
results = model.fit
print(results.summary())

In [ ]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [ ]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rsme(df["Home_Value"],y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

# ── 1. EXAMPLE DATA (swap in your own df, y, X) ──────────────────────────────
# Using the Boston Housing dataset as a reproducible stand-in
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['Price'] = housing.target

# Define your features and target
X_raw = df[['MedInc', 'HouseAge', 'AveRooms', 'AveOccup']]
y     = df['Price']

# ── 2. FIT THE OLS MODEL ─────────────────────────────────────────────────────
X = sm.add_constant(X_raw)          # statsmodels requires explicit intercept column
model   = sm.OLS(y, X).fit()        # .fit() returns a RegressionResultsWrapper object

print(model.summary())

# ── 3. EXTRACT RESIDUALS & FITTED VALUES FROM RESULTS OBJECT ─────────────────
# model.fittedvalues → pandas Series of ŷ (X @ β̂), aligned to original index
# model.resid        → pandas Series of ε̂ = y - ŷ, same index
fitted    = model.fittedvalues
residuals = model.resid

# ── 4. RMSE ───────────────────────────────────────────────────────────────────
rmse = np.sqrt(np.mean(residuals ** 2))
print(f"\nRMSE: {rmse:.4f}")

# ── 5. FLAG OUTLIERS (> ±2 SD) ───────────────────────────────────────────────
resid_std   = residuals.std()                          # 1 SD of the residual distribution
resid_mean  = residuals.mean()                         # should be ≈ 0 in OLS by construction

# Boolean mask: True where the residual exceeds the 2-SD envelope
is_outlier  = (residuals - resid_mean).abs() > 2 * resid_std

# Map boolean to a human-readable label for the color channel
outlier_label = is_outlier.map({True: 'Outlier (>2σ)', False: 'Normal'})

# ── 6. ASSEMBLE PLOT DATAFRAME ────────────────────────────────────────────────
plot_df = pd.DataFrame({
    'Fitted Values' : fitted,
    'Residuals'     : residuals,
    'Point Type'    : outlier_label
})

# ── 7. BUILD THE INTERACTIVE RESIDUALS SCATTER ───────────────────────────────
fig = px.scatter(
    plot_df,
    x            = 'Fitted Values',
    y            = 'Residuals',
    color        = 'Point Type',
    color_discrete_map = {
        'Normal'        : 'steelblue',
        'Outlier (>2σ)' : 'crimson'      # stark crimson for outliers
    },
    opacity      = 0.65,
    title        = 'Residual Forensics Dashboard — OLS Hedonic Pricing Model',
    labels       = {'Fitted Values': 'Fitted Values (ŷ)', 'Residuals': 'Residuals (ε̂)'},
    hover_data   = {'Fitted Values': ':.4f', 'Residuals': ':.4f'}
)

# ── 8. ZERO-LINE + ENVELOPE BANDS ────────────────────────────────────────────
# Horizontal zero-line: the theoretical residual mean under OLS assumptions
fig.add_hline(
    y           = 0,
    line_dash   = 'solid',
    line_color  = 'black',
    line_width  = 1.5,
    annotation_text = 'ε = 0'
)

# +2σ band (upper threshold)
fig.add_hline(
    y           = resid_mean + 2 * resid_std,
    line_dash   = 'dash',
    line_color  = 'crimson',
    line_width  = 1,
    annotation_text = '+2σ'
)

# -2σ band (lower threshold)
fig.add_hline(
    y           = resid_mean - 2 * resid_std,
    line_dash   = 'dash',
    line_color  = 'crimson',
    line_width  = 1,
    annotation_text = '−2σ'
)

fig.update_layout(
    plot_bgcolor  = 'white',
    paper_bgcolor = 'white',
    xaxis         = dict(showgrid=True, gridcolor='lightgrey'),
    yaxis         = dict(showgrid=True, gridcolor='lightgrey'),
    legend_title  = 'Point Classification',
    title_font    = dict(size=18)
)

fig.show()